# Monte Carlo Simulation of Equal Weighted Portfolios

This notebook simulates portfolios of $N$ independent assets that are Gaussian distributed. 
We assume an annualized mean return of 10% and an annualized standard deviation of 18%. 
For each $N$, we simulate 1000 trials of a 10-year period (120 months) and calculate the median, 25th, and 75th percentiles for the annualized mean return, annualized volatility, and maximum drawdown.

In [3]:
import numpy as np
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots

def calc_mdd(returns):
    cum_returns = (1 + returns).cumprod()
    running_max = np.maximum.accumulate(cum_returns)
    drawdown = (cum_returns - running_max) / running_max
    return np.min(drawdown)

# Parameters
annual_mean = 0.09
annual_std = 0.18
monthly_mean = annual_mean / 12
monthly_std = annual_std / np.sqrt(12)

T_months = 120 # 10 years of monthly returns per simulation
num_sims = 1000
N_values = [i for i in range(15)]

results = {
    'N': N_values,
    'mean_p50': [], 'mean_p25': [], 'mean_p75': [],
    'std_p50': [], 'std_p25': [], 'std_p75': [],
    'mdd_p50': [], 'mdd_p25': [], 'mdd_p75': []
}

print(f"Running Monte Carlo Simulation ({num_sims} iterations per N)...")

for N in N_values:
    means = np.zeros(num_sims)
    stds = np.zeros(num_sims)
    mdds = np.zeros(num_sims)
    
    for i in range(num_sims):
        # Simulate T_months returns for N independent assets
        asset_returns = np.random.normal(monthly_mean, monthly_std, (T_months, N))
        
        # Equal weight portfolio return is the mean across assets for each month
        port_returns = asset_returns.mean(axis=1)
        
        # Calculate annualized statistics
        ann_mean = port_returns.mean() * 12 * 100
        ann_std = port_returns.std() * np.sqrt(12) * 100
        mdd = calc_mdd(port_returns) * 100
        
        means[i] = ann_mean
        stds[i] = ann_std
        mdds[i] = mdd
        
    # Record statistics
    results['mean_p50'].append(np.median(means))
    results['mean_p25'].append(np.percentile(means, 25))
    results['mean_p75'].append(np.percentile(means, 75))
    
    results['std_p50'].append(np.median(stds))
    results['std_p25'].append(np.percentile(stds, 25))
    results['std_p75'].append(np.percentile(stds, 75))
    
    results['mdd_p50'].append(np.median(mdds))
    results['mdd_p25'].append(np.percentile(mdds, 25))
    results['mdd_p75'].append(np.percentile(mdds, 75))


Running Monte Carlo Simulation (1000 iterations per N)...


/var/folders/mc/qf75k40s6ns_nr8c35wdmp400000gn/T/ipykernel_36234/2492389055.py:41: RuntimeWarning: Mean of empty slice
  port_returns = asset_returns.mean(axis=1)
/Users/arthurdhonneur/Desktop/Athenee/athenee_hf_pflio/.venv/lib/python3.13/site-packages/numpy/_core/_methods.py:134: RuntimeWarning: invalid value encountered in divide
  ret = um.true_divide(


In [4]:
# Create subplots
fig = make_subplots(rows=3, cols=1, 
                    subplot_titles=('Annualized Mean Return vs Number of Assets (N)',
                                    'Annualized Volatility (Std Dev) vs Number of Assets (N)',
                                    'Maximum Drawdown vs Number of Assets (N)'),
                    vertical_spacing=0.1)

# 1. Mean Return
fig.add_trace(go.Scatter(x=results['N'], y=results['mean_p50'], mode='lines+markers', name='Median Mean', line=dict(color='blue'), hovertemplate='N: %{x}<br>Mean: %{y:.4f}<extra></extra>'), row=1, col=1)
fig.add_trace(go.Scatter(x=results['N'], y=results['mean_p75'], mode='lines', line=dict(width=0), showlegend=False, hoverinfo='skip'), row=1, col=1)
fig.add_trace(go.Scatter(x=results['N'], y=results['mean_p25'], mode='lines', fill='tonexty', fillcolor='rgba(0,0,255,0.2)', line=dict(width=0), name='IQR (Mean)', hovertemplate='N: %{x}<br>25th-75th IQR<extra></extra>'), row=1, col=1)
fig.add_hline(y=annual_mean, line_dash="dash", line_color="black", annotation_text="True Mean (10%)", row=1, col=1)

# 2. Volatility
fig.add_trace(go.Scatter(x=results['N'], y=results['std_p50'], mode='lines+markers', name='Median Volatility', line=dict(color='orange'), hovertemplate='N: %{x}<br>Vol: %{y:.4f}<extra></extra>'), row=2, col=1)
fig.add_trace(go.Scatter(x=results['N'], y=results['std_p75'], mode='lines', line=dict(width=0), showlegend=False, hoverinfo='skip'), row=2, col=1)
fig.add_trace(go.Scatter(x=results['N'], y=results['std_p25'], mode='lines', fill='tonexty', fillcolor='rgba(255,165,0,0.2)', line=dict(width=0), name='IQR (Volatility)', hovertemplate='N: %{x}<br>25th-75th IQR<extra></extra>'), row=2, col=1)

# 3. Maximum Drawdown
fig.add_trace(go.Scatter(x=results['N'], y=results['mdd_p50'], mode='lines+markers', name='Median MDD', line=dict(color='red'), hovertemplate='N: %{x}<br>MDD: %{y:.4f}<extra></extra>'), row=3, col=1)
fig.add_trace(go.Scatter(x=results['N'], y=results['mdd_p75'], mode='lines', line=dict(width=0), showlegend=False, hoverinfo='skip'), row=3, col=1)
fig.add_trace(go.Scatter(x=results['N'], y=results['mdd_p25'], mode='lines', fill='tonexty', fillcolor='rgba(255,0,0,0.2)', line=dict(width=0), name='IQR (MDD)', hovertemplate='N: %{x}<br>25th-75th IQR<extra></extra>'), row=3, col=1)

# Update layout
fig.update_layout(height=1000, width=800, title_text="Monte Carlo Simulation Results", showlegend=True, hovermode='x unified')
fig.update_xaxes(title_text="Number of Assets (N)", row=3, col=1)

# Set common labels
fig.update_yaxes(title_text="Mean Return", row=1, col=1)
fig.update_yaxes(title_text="Volatility", row=2, col=1)
fig.update_yaxes(title_text="Max Drawdown", row=3, col=1)

fig.show()
